# **<span style="color:blue">Notebook 7 - Prof. Dr. Humberto Elias Lopes - Teste t para duas Amostras Independentes</span>**

## Revisão rápida dos conceitos

Duas amostras são **independentes** quando não existe ligação entre os elementos de um grupo e os elementos do outro. É o caso de clientes de duas lojas diferentes, pedidos de dois fornecedores ou funcionários de dois regimes de trabalho. Os grupos podem até ter tamanhos diferentes.

Na aula passada, no teste pareado, os **mesmos** elementos eram medidos duas vezes, e por isso trabalhávamos com as diferenças de cada par. Agora não há pares: comparamos diretamente a média de um grupo com a média do outro.

As hipóteses são formuladas sobre a diferença entre as médias populacionais, $\mu_1 - \mu_2$:

| Pergunta gerencial | $H_0$ | $H_1$ | Argumento `alternative` |
|---|---|---|---|
| As médias são **diferentes**? | $\mu_1 - \mu_2 = 0$ | $\mu_1 - \mu_2 \neq 0$ | `'two-sided'` |
| A média do grupo 1 é **maior**? | $\mu_1 - \mu_2 \le 0$ | $\mu_1 - \mu_2 > 0$ | `'greater'` |
| A média do grupo 1 é **menor**? | $\mu_1 - \mu_2 \ge 0$ | $\mu_1 - \mu_2 < 0$ | `'less'` |

A estatística de teste é

$$t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\dfrac{s_1^2}{n_1} + \dfrac{s_2^2}{n_2}}},$$

em que $\bar{x}_1$ e $\bar{x}_2$ são as médias amostrais, $s_1$ e $s_2$ são os desvios-padrão amostrais e $n_1$ e $n_2$ são os tamanhos das amostras. O denominador é o **erro-padrão** da diferença entre as médias.

**Graus de liberdade.** Como não supomos que as duas populações tenham a mesma variância, os graus de liberdade são calculados por uma fórmula mais longa (aproximação de Welch-Satterthwaite) e, em geral, não são um número inteiro. Não vamos calculá-los à mão: o Python fará essa conta por nós.

**Pressupostos.** As duas amostras são aleatórias e independentes entre si, e cada população tem distribuição aproximadamente normal. Com amostras pequenas, como as desta aula, o segundo pressuposto é importante.

**Regra de decisão.** Fixado o nível de significância $\alpha$ (em geral, 5%), rejeitamos $H_0$ quando o valor-p for menor que $\alpha$.

## Passo zero: um pouco de Python

Nesta aula, vamos usar apenas estes recursos da linguagem:

* **listas**, que guardam vários valores entre colchetes: `[10, 20, 30]`;
* a função `len()`, que conta quantos itens há em uma lista;
* a função `print()`, que mostra resultados na tela;
* o operador `**`, que eleva um número a uma potência: `3 ** 2` é 3 ao quadrado.

Diferentemente da aula passada, não precisaremos do laço `for`, porque não há diferenças par a par para calcular. Execute a célula abaixo para testar.

In [ ]:
vendas = [10, 20, 30] # Cria uma lista com três valores.
print(vendas) # Mostra a lista.
print(len(vendas)) # Mostra quantos itens há na lista.
print(3 ** 2) # Mostra 3 elevado ao quadrado.

Agora vamos carregar as ferramentas que usaremos na aula:

* `math`, para a raiz quadrada;
* `statistics`, para a média e o desvio-padrão;
* `stats`, da biblioteca SciPy, para o teste t e a distribuição t.

In [2]:
import math # Raiz quadrada.
import statistics # Média e desvio-padrão.
from scipy import stats # Teste t e distribuição t.

---
## Exemplo 1. Compras pelo aplicativo e na loja física

Uma rede de livrarias quer saber se os clientes que compram pelo **aplicativo** gastam mais por compra do que os clientes que compram na **loja física**. Foram sorteadas 10 compras feitas no aplicativo e 9 compras feitas na loja, e registrado o valor de cada uma, em reais.

| Grupo | n | Valores |
|---|---|---|
| Aplicativo | 10 | 182; 195; 170; 210; 188; 176; 201; 192; 185; 199 |
| Loja física | 9 | 160; 172; 155; 181; 168; 149; 175; 163; 170 |

**Pergunta gerencial.** Os clientes do aplicativo gastam, em média, **mais** por compra?

**Hipóteses** (grupo 1 = aplicativo; grupo 2 = loja física):

* $H_0$: $\mu_{app} - \mu_{loja} \le 0$ (o gasto médio no aplicativo não é maior)
* $H_1$: $\mu_{app} - \mu_{loja} > 0$ (o gasto médio no aplicativo é maior)

Teste unilateral à direita, com nível de significância $\alpha = 0{,}05$.

**Passo 1.** Digitar os dados em duas listas, uma para cada grupo. As listas podem ter tamanhos diferentes.

In [ ]:
app = [182, 195, 170, 210, 188, 176, 201, 192, 185, 199]
loja = [160, 172, 155, 181, 168, 149, 175, 163, 170]

print("Média de app:", round(statistics.mean(app), 4))
print("Média de loja:", round(statistics.mean(loja), 4))

**Passo 2.** Calcular o tamanho, a média e o desvio-padrão de cada grupo. Usamos o número 1 para o primeiro grupo e o número 2 para o segundo, como na fórmula.

In [ ]:
n1 = len(app) # Tamanho do grupo 1.
n2 = len(loja) # Tamanho do grupo 2.
media1 = statistics.mean(app) # Média do grupo 1.
media2 = statistics.mean(loja) # Média do grupo 2.
dp1 = statistics.stdev(app) # Desvio-padrão amostral do grupo 1.
dp2 = statistics.stdev(loja) # Desvio-padrão amostral do grupo 2.

print("Grupo 1: n =", n1, "| média =", round(media1, 4), "| desvio-padrão =", round(dp1, 4))
print("Grupo 2: n =", n2, "| média =", round(media2, 4), "| desvio-padrão =", round(dp2, 4))

**Passo 3.** Calcular a estatística t à mão, usando a fórmula.

In [ ]:
diferenca = media1 - media2 # Diferença entre as médias amostrais.
erro_padrao = math.sqrt(dp1 ** 2 / n1 + dp2 ** 2 / n2) # Erro-padrão da diferença.
t_calculado = diferenca / erro_padrao # Estatística t.

print("Diferença entre as médias:", round(diferenca, 4))
print("Erro-padrão:", round(erro_padrao, 4))
print("t calculado:", round(t_calculado, 4))

**Passo 4.** Conferir com a função `ttest_ind`, que já faz tudo de uma vez. A primeira lista informada é `app` (grupo 1) e a segunda é `loja` (grupo 2). O argumento `equal_var=False` indica que não supomos variâncias iguais nos dois grupos, e o resultado traz também os graus de liberdade (`resultado.df`).

In [ ]:
resultado = stats.ttest_ind(app, loja, equal_var=False, alternative='greater')

print("t (SciPy):", round(resultado.statistic, 4))
print("valor-p:", round(resultado.pvalue, 4))
print("Graus de liberdade:", round(resultado.df, 2))

**Passo 5.** Comparar com o valor crítico e tomar a decisão.

In [ ]:
alfa = 0.05
t_critico = stats.t.ppf(1 - alfa, resultado.df) # Teste unilateral à direita.
print("t crítico:", round(t_critico, 4))

if resultado.pvalue < alfa:
    print("Decisão: rejeitamos H0.")
else:
    print("Decisão: não rejeitamos H0.")

**Interpretação gerencial.** A compra média no aplicativo foi de R\$ 189,80, contra R\$ 165,89 na loja física, uma diferença de R\$ 23,91 (t = 4,69; valor-p ≈ 0,0001). Como o valor-p é muito menor que 5%, rejeitamos $H_0$: há evidência de que os clientes do aplicativo gastam mais por compra. O resultado justifica investir na divulgação do aplicativo, mas não prova que o aplicativo **causa** o gasto maior. Pode ser que os clientes que preferem o aplicativo já tenham um perfil de consumo mais alto. Para medir o efeito do canal, a empresa precisaria de um experimento.

---
## Exemplo 2. Prazo de entrega de dois fornecedores

Uma fábrica de móveis compra chapas de MDF de dois fornecedores. O gerente de suprimentos registrou o prazo de entrega, em dias, de 8 pedidos feitos ao fornecedor A e de 10 pedidos feitos ao fornecedor B.

| Grupo | n | Valores |
|---|---|---|
| Fornecedor A | 8 | 12; 15; 11; 14; 13; 16; 12; 14 |
| Fornecedor B | 10 | 9; 11; 10; 12; 8; 10; 11; 9; 10; 12 |

**Pergunta gerencial.** Os prazos médios de entrega dos dois fornecedores são **diferentes**?

**Hipóteses** (grupo 1 = fornecedor a; grupo 2 = fornecedor b):

* $H_0$: $\mu_A - \mu_B = 0$ (os prazos médios são iguais)
* $H_1$: $\mu_A - \mu_B \neq 0$ (os prazos médios são diferentes)

Teste bilateral, com nível de significância $\alpha = 0{,}05$.

**Passo 1.** Digitar os dados em duas listas, uma para cada grupo. As listas podem ter tamanhos diferentes.

In [8]:
fornecedor_a = [12, 15, 11, 14, 13, 16, 12, 14]
fornecedor_b = [9, 11, 10, 12, 8, 10, 11, 9, 10, 12]

print("Média de fornecedor_a:", round(statistics.mean(fornecedor_a), 4))
print("Média de fornecedor_b:", round(statistics.mean(fornecedor_b), 4))

Média de fornecedor_a: 13.375
Média de fornecedor_b: 10.2


**Passo 2.** Calcular o tamanho, a média e o desvio-padrão de cada grupo.

In [9]:
n1 = len(fornecedor_a) # Tamanho do grupo 1.
n2 = len(fornecedor_b) # Tamanho do grupo 2.
media1 = statistics.mean(fornecedor_a) # Média do grupo 1.
media2 = statistics.mean(fornecedor_b) # Média do grupo 2.
dp1 = statistics.stdev(fornecedor_a) # Desvio-padrão amostral do grupo 1.
dp2 = statistics.stdev(fornecedor_b) # Desvio-padrão amostral do grupo 2.

print("Grupo 1: n =", n1, "| média =", round(media1, 4), "| desvio-padrão =", round(dp1, 4))
print("Grupo 2: n =", n2, "| média =", round(media2, 4), "| desvio-padrão =", round(dp2, 4))

Grupo 1: n = 8 | média = 13.375 | desvio-padrão = 1.685
Grupo 2: n = 10 | média = 10.2 | desvio-padrão = 1.3166


**Passo 3.** Calcular a estatística t à mão, usando a fórmula.

In [10]:
diferenca = media1 - media2 # Diferença entre as médias amostrais.
erro_padrao = math.sqrt(dp1 ** 2 / n1 + dp2 ** 2 / n2) # Erro-padrão da diferença.
t_calculado = diferenca / erro_padrao # Estatística t.

print("Diferença entre as médias:", round(diferenca, 4))
print("Erro-padrão:", round(erro_padrao, 4))
print("t calculado:", round(t_calculado, 4))

Diferença entre as médias: 3.175
Erro-padrão: 0.7268
t calculado: 4.3684


**Passo 4.** Conferir com a função `ttest_ind`, que já faz tudo de uma vez. A primeira lista informada é `fornecedor_a` (grupo 1) e a segunda é `fornecedor_b` (grupo 2).

In [11]:
resultado = stats.ttest_ind(fornecedor_a, fornecedor_b, equal_var=False, alternative='two-sided')

print("t (SciPy):", round(resultado.statistic, 4))
print("valor-p:", round(resultado.pvalue, 4))
print("Graus de liberdade:", round(resultado.df, 2))

t (SciPy): 4.3684
valor-p: 0.0007
Graus de liberdade: 13.08


**Passo 5.** Comparar com o valor crítico e tomar a decisão.

In [12]:
alfa = 0.05
t_critico = stats.t.ppf(1 - alfa / 2, resultado.df) # Teste bilateral: alfa dividido entre as duas caudas.
print("t crítico:", round(t_critico, 4))

if resultado.pvalue < alfa:
    print("Decisão: rejeitamos H0.")
else:
    print("Decisão: não rejeitamos H0.")

t crítico: 2.159
Decisão: rejeitamos H0.


**Interpretação gerencial.** O fornecedor A levou, em média, 13,38 dias para entregar, e o fornecedor B, 10,20 dias, uma diferença de 3,18 dias (t = 4,37; valor-p ≈ 0,0007). Rejeitamos $H_0$: os prazos médios são diferentes, e o fornecedor B é mais rápido. O teste é bilateral porque, antes de coletar os dados, o gerente não tinha uma suspeita sobre qual fornecedor seria mais rápido. Antes de concentrar as compras no fornecedor B, ele deve comparar também preço e qualidade das chapas.

---
## Exemplo 3. Tempo de espera em duas agências bancárias

Um banco implantou uma triagem digital na entrada da agência Centro, mas não na agência Savassi. O gerente regional registrou o tempo de espera, em minutos, de 11 clientes da agência Centro e de 9 clientes da agência Savassi, em dias de movimento semelhante.

| Grupo | n | Valores |
|---|---|---|
| Centro (com triagem) | 11 | 6,2; 5,8; 7,1; 6,5; 5,9; 6,8; 6,0; 6,4; 5,7; 6,6; 6,1 |
| Savassi (sem triagem) | 9 | 7,4; 6,3; 8,1; 6,5; 7,8; 6,2; 6,6; 7,5; 7,9 |

**Pergunta gerencial.** A agência com triagem digital tem tempo médio de espera **menor**?

**Hipóteses** (grupo 1 = centro (com triagem); grupo 2 = savassi (sem triagem)):

* $H_0$: $\mu_{centro} - \mu_{savassi} \ge 0$ (a espera na agência com triagem não é menor)
* $H_1$: $\mu_{centro} - \mu_{savassi} < 0$ (a espera na agência com triagem é menor)

Teste unilateral à esquerda, com nível de significância $\alpha = 0{,}05$.

**Passo 1.** Digitar os dados em duas listas, uma para cada grupo. As listas podem ter tamanhos diferentes. Lembre-se de que, em Python, o separador decimal é o ponto.

In [13]:
centro = [6.2, 5.8, 7.1, 6.5, 5.9, 6.8, 6.0, 6.4, 5.7, 6.6, 6.1]
savassi = [7.4, 6.3, 8.1, 6.5, 7.8, 6.2, 6.6, 7.5, 7.9]

print("Média de centro:", round(statistics.mean(centro), 4))
print("Média de savassi:", round(statistics.mean(savassi), 4))

Média de centro: 6.2818
Média de savassi: 7.1444


**Passo 2.** Calcular o tamanho, a média e o desvio-padrão de cada grupo.

In [14]:
n1 = len(centro) # Tamanho do grupo 1.
n2 = len(savassi) # Tamanho do grupo 2.
media1 = statistics.mean(centro) # Média do grupo 1.
media2 = statistics.mean(savassi) # Média do grupo 2.
dp1 = statistics.stdev(centro) # Desvio-padrão amostral do grupo 1.
dp2 = statistics.stdev(savassi) # Desvio-padrão amostral do grupo 2.

print("Grupo 1: n =", n1, "| média =", round(media1, 4), "| desvio-padrão =", round(dp1, 4))
print("Grupo 2: n =", n2, "| média =", round(media2, 4), "| desvio-padrão =", round(dp2, 4))

Grupo 1: n = 11 | média = 6.2818 | desvio-padrão = 0.44
Grupo 2: n = 9 | média = 7.1444 | desvio-padrão = 0.7435


**Passo 3.** Calcular a estatística t à mão, usando a fórmula.

In [15]:
diferenca = media1 - media2 # Diferença entre as médias amostrais.
erro_padrao = math.sqrt(dp1 ** 2 / n1 + dp2 ** 2 / n2) # Erro-padrão da diferença.
t_calculado = diferenca / erro_padrao # Estatística t.

print("Diferença entre as médias:", round(diferenca, 4))
print("Erro-padrão:", round(erro_padrao, 4))
print("t calculado:", round(t_calculado, 4))

Diferença entre as médias: -0.8626
Erro-padrão: 0.2811
t calculado: -3.0686


**Passo 4.** Conferir com a função `ttest_ind`, que já faz tudo de uma vez. A primeira lista informada é `centro` (grupo 1) e a segunda é `savassi` (grupo 2).

In [16]:
resultado = stats.ttest_ind(centro, savassi, equal_var=False, alternative='less')

print("t (SciPy):", round(resultado.statistic, 4))
print("valor-p:", round(resultado.pvalue, 4))
print("Graus de liberdade:", round(resultado.df, 2))

t (SciPy): -3.0686
valor-p: 0.0047
Graus de liberdade: 12.43


**Passo 5.** Comparar com o valor crítico e tomar a decisão.

In [17]:
alfa = 0.05
t_critico = stats.t.ppf(alfa, resultado.df) # Teste unilateral à esquerda: o valor crítico é negativo.
print("t crítico:", round(t_critico, 4))

if resultado.pvalue < alfa:
    print("Decisão: rejeitamos H0.")
else:
    print("Decisão: não rejeitamos H0.")

t crítico: -1.7772
Decisão: rejeitamos H0.


**Interpretação gerencial.** O tempo médio de espera foi de 6,28 minutos na agência Centro e de 7,14 minutos na Savassi, uma diferença de 0,86 minuto (t = −3,07; valor-p ≈ 0,0047). Rejeitamos $H_0$: há evidência de que a espera é menor na agência com triagem. Note que o t é negativo porque a média do primeiro grupo é menor, o que está de acordo com $H_1$. Como as agências diferem em outros aspectos (localização, número de caixas, perfil dos clientes), o banco deve testar a triagem em mais agências antes de atribuir a ela toda a diferença.

---
## Exemplo 4. Satisfação de funcionários remotos e presenciais

Uma empresa de software aplicou uma pesquisa de satisfação, com notas de 0 a 10, a 10 funcionários em trabalho remoto e a 12 funcionários em trabalho presencial.

| Grupo | n | Valores |
|---|---|---|
| Remoto | 10 | 7,5; 8,0; 6,5; 7,0; 8,5; 6,0; 7,5; 8,0; 7,0; 6,5 |
| Presencial | 12 | 7,0; 7,5; 8,0; 6,5; 7,0; 7,5; 6,0; 8,5; 7,0; 7,5; 6,5; 7,0 |

**Pergunta gerencial.** A satisfação média dos dois grupos é **diferente**?

**Hipóteses** (grupo 1 = remoto; grupo 2 = presencial):

* $H_0$: $\mu_{remoto} - \mu_{presencial} = 0$ (a satisfação média é igual nos dois regimes)
* $H_1$: $\mu_{remoto} - \mu_{presencial} \neq 0$ (a satisfação média é diferente)

Teste bilateral, com nível de significância $\alpha = 0{,}05$.

**Passo 1.** Digitar os dados em duas listas, uma para cada grupo. As listas podem ter tamanhos diferentes.

In [18]:
remoto = [7.5, 8.0, 6.5, 7.0, 8.5, 6.0, 7.5, 8.0, 7.0, 6.5]
presencial = [7.0, 7.5, 8.0, 6.5, 7.0, 7.5, 6.0, 8.5, 7.0, 7.5, 6.5, 7.0]

print("Média de remoto:", round(statistics.mean(remoto), 4))
print("Média de presencial:", round(statistics.mean(presencial), 4))

Média de remoto: 7.25
Média de presencial: 7.1667


**Passo 2.** Calcular o tamanho, a média e o desvio-padrão de cada grupo.

In [19]:
n1 = len(remoto) # Tamanho do grupo 1.
n2 = len(presencial) # Tamanho do grupo 2.
media1 = statistics.mean(remoto) # Média do grupo 1.
media2 = statistics.mean(presencial) # Média do grupo 2.
dp1 = statistics.stdev(remoto) # Desvio-padrão amostral do grupo 1.
dp2 = statistics.stdev(presencial) # Desvio-padrão amostral do grupo 2.

print("Grupo 1: n =", n1, "| média =", round(media1, 4), "| desvio-padrão =", round(dp1, 4))
print("Grupo 2: n =", n2, "| média =", round(media2, 4), "| desvio-padrão =", round(dp2, 4))

Grupo 1: n = 10 | média = 7.25 | desvio-padrão = 0.7906
Grupo 2: n = 12 | média = 7.1667 | desvio-padrão = 0.6853


**Passo 3.** Calcular a estatística t à mão, usando a fórmula.

In [20]:
diferenca = media1 - media2 # Diferença entre as médias amostrais.
erro_padrao = math.sqrt(dp1 ** 2 / n1 + dp2 ** 2 / n2) # Erro-padrão da diferença.
t_calculado = diferenca / erro_padrao # Estatística t.

print("Diferença entre as médias:", round(diferenca, 4))
print("Erro-padrão:", round(erro_padrao, 4))
print("t calculado:", round(t_calculado, 4))

Diferença entre as médias: 0.0833
Erro-padrão: 0.3188
t calculado: 0.2614


**Passo 4.** Conferir com a função `ttest_ind`, que já faz tudo de uma vez. A primeira lista informada é `remoto` (grupo 1) e a segunda é `presencial` (grupo 2).

In [21]:
resultado = stats.ttest_ind(remoto, presencial, equal_var=False, alternative='two-sided')

print("t (SciPy):", round(resultado.statistic, 4))
print("valor-p:", round(resultado.pvalue, 4))
print("Graus de liberdade:", round(resultado.df, 2))

t (SciPy): 0.2614
valor-p: 0.7968
Graus de liberdade: 18.02


**Passo 5.** Comparar com o valor crítico e tomar a decisão.

In [22]:
alfa = 0.05
t_critico = stats.t.ppf(1 - alfa / 2, resultado.df) # Teste bilateral: alfa dividido entre as duas caudas.
print("t crítico:", round(t_critico, 4))

if resultado.pvalue < alfa:
    print("Decisão: rejeitamos H0.")
else:
    print("Decisão: não rejeitamos H0.")

t crítico: 2.1008
Decisão: não rejeitamos H0.


**Interpretação gerencial.** A nota média foi de 7,25 no trabalho remoto e de 7,17 no presencial, uma diferença de apenas 0,08 ponto (t = 0,26; valor-p ≈ 0,80). **Não rejeitamos** $H_0$: os dados não mostram diferença de satisfação entre os regimes. Isso não prova que a satisfação seja exatamente igual, apenas que a diferença observada é compatível com o acaso. Para a área de pessoas, o resultado sugere que o regime de trabalho não é, por si só, o principal fator de satisfação nessa empresa.

---
## Exemplo 5. Nova embalagem de suco

Um fabricante de sucos lançou uma nova embalagem em 8 supermercados, enquanto outros 8 supermercados, de porte semelhante, continuaram com a embalagem atual. Foram registradas as vendas diárias médias, em unidades, de cada supermercado.

| Grupo | n | Valores |
|---|---|---|
| Embalagem nova | 8 | 48; 52; 45; 50; 55; 47; 51; 49 |
| Embalagem atual | 8 | 46; 50; 47; 44; 52; 48; 45; 49 |

**Pergunta gerencial.** A nova embalagem **aumentou** as vendas médias?

**Hipóteses** (grupo 1 = embalagem nova; grupo 2 = embalagem atual):

* $H_0$: $\mu_{nova} - \mu_{atual} \le 0$ (a nova embalagem não vende mais)
* $H_1$: $\mu_{nova} - \mu_{atual} > 0$ (a nova embalagem vende mais)

Teste unilateral à direita, com nível de significância $\alpha = 0{,}05$.

**Passo 1.** Digitar os dados em duas listas, uma para cada grupo. As listas podem ter tamanhos diferentes.

In [23]:
nova = [48, 52, 45, 50, 55, 47, 51, 49]
atual = [46, 50, 47, 44, 52, 48, 45, 49]

print("Média de nova:", round(statistics.mean(nova), 4))
print("Média de atual:", round(statistics.mean(atual), 4))

Média de nova: 49.625
Média de atual: 47.625


**Passo 2.** Calcular o tamanho, a média e o desvio-padrão de cada grupo.

In [24]:
n1 = len(nova) # Tamanho do grupo 1.
n2 = len(atual) # Tamanho do grupo 2.
media1 = statistics.mean(nova) # Média do grupo 1.
media2 = statistics.mean(atual) # Média do grupo 2.
dp1 = statistics.stdev(nova) # Desvio-padrão amostral do grupo 1.
dp2 = statistics.stdev(atual) # Desvio-padrão amostral do grupo 2.

print("Grupo 1: n =", n1, "| média =", round(media1, 4), "| desvio-padrão =", round(dp1, 4))
print("Grupo 2: n =", n2, "| média =", round(media2, 4), "| desvio-padrão =", round(dp2, 4))

Grupo 1: n = 8 | média = 49.625 | desvio-padrão = 3.1139
Grupo 2: n = 8 | média = 47.625 | desvio-padrão = 2.6693


**Passo 3.** Calcular a estatística t à mão, usando a fórmula.

In [25]:
diferenca = media1 - media2 # Diferença entre as médias amostrais.
erro_padrao = math.sqrt(dp1 ** 2 / n1 + dp2 ** 2 / n2) # Erro-padrão da diferença.
t_calculado = diferenca / erro_padrao # Estatística t.

print("Diferença entre as médias:", round(diferenca, 4))
print("Erro-padrão:", round(erro_padrao, 4))
print("t calculado:", round(t_calculado, 4))

Diferença entre as médias: 2.0
Erro-padrão: 1.4501
t calculado: 1.3793


**Passo 4.** Conferir com a função `ttest_ind`, que já faz tudo de uma vez. A primeira lista informada é `nova` (grupo 1) e a segunda é `atual` (grupo 2).

In [26]:
resultado = stats.ttest_ind(nova, atual, equal_var=False, alternative='greater')

print("t (SciPy):", round(resultado.statistic, 4))
print("valor-p:", round(resultado.pvalue, 4))
print("Graus de liberdade:", round(resultado.df, 2))

t (SciPy): 1.3793
valor-p: 0.095
Graus de liberdade: 13.68


**Passo 5.** Comparar com o valor crítico e tomar a decisão.

In [27]:
alfa = 0.05
t_critico = stats.t.ppf(1 - alfa, resultado.df) # Teste unilateral à direita.
print("t crítico:", round(t_critico, 4))

if resultado.pvalue < alfa:
    print("Decisão: rejeitamos H0.")
else:
    print("Decisão: não rejeitamos H0.")

t crítico: 1.7642
Decisão: não rejeitamos H0.


**Interpretação gerencial.** Os supermercados com a embalagem nova venderam, em média, 49,63 unidades por dia, contra 47,63 dos demais, uma diferença de 2 unidades (t = 1,38; valor-p ≈ 0,095). Ao nível de 5%, **não rejeitamos** $H_0$. A diferença vai na direção esperada, mas é pequena diante da variação entre os supermercados. Antes de trocar a embalagem em toda a rede, o fabricante deveria ampliar o teste para mais lojas ou por mais tempo.

---
## Exemplo 6. Produtividade de duas máquinas

Uma metalúrgica avalia duas máquinas de estampagem, de fabricantes diferentes, para decidir qual comprar. Cada máquina foi testada em 8 turnos diferentes, e foi registrado o número de peças produzidas por hora.

| Grupo | n | Valores |
|---|---|---|
| Máquina X | 8 | 118; 124; 121; 115; 127; 119; 123; 120 |
| Máquina Y | 8 | 114; 120; 117; 116; 114; 122; 113; 121 |

**Pergunta gerencial.** A produtividade média das duas máquinas é **diferente**?

**Hipóteses** (grupo 1 = máquina x; grupo 2 = máquina y):

* $H_0$: $\mu_X - \mu_Y = 0$ (as máquinas têm a mesma produtividade média)
* $H_1$: $\mu_X - \mu_Y \neq 0$ (as produtividades médias são diferentes)

Teste bilateral, com nível de significância $\alpha = 0{,}05$.

**Passo 1.** Digitar os dados em duas listas, uma para cada grupo. As listas podem ter tamanhos diferentes.

In [28]:
maquina_x = [118, 124, 121, 115, 127, 119, 123, 120]
maquina_y = [114, 120, 117, 116, 114, 122, 113, 121]

print("Média de maquina_x:", round(statistics.mean(maquina_x), 4))
print("Média de maquina_y:", round(statistics.mean(maquina_y), 4))

Média de maquina_x: 120.875
Média de maquina_y: 117.125


**Passo 2.** Calcular o tamanho, a média e o desvio-padrão de cada grupo.

In [29]:
n1 = len(maquina_x) # Tamanho do grupo 1.
n2 = len(maquina_y) # Tamanho do grupo 2.
media1 = statistics.mean(maquina_x) # Média do grupo 1.
media2 = statistics.mean(maquina_y) # Média do grupo 2.
dp1 = statistics.stdev(maquina_x) # Desvio-padrão amostral do grupo 1.
dp2 = statistics.stdev(maquina_y) # Desvio-padrão amostral do grupo 2.

print("Grupo 1: n =", n1, "| média =", round(media1, 4), "| desvio-padrão =", round(dp1, 4))
print("Grupo 2: n =", n2, "| média =", round(media2, 4), "| desvio-padrão =", round(dp2, 4))

Grupo 1: n = 8 | média = 120.875 | desvio-padrão = 3.7583
Grupo 2: n = 8 | média = 117.125 | desvio-padrão = 3.4821


**Passo 3.** Calcular a estatística t à mão, usando a fórmula.

In [30]:
diferenca = media1 - media2 # Diferença entre as médias amostrais.
erro_padrao = math.sqrt(dp1 ** 2 / n1 + dp2 ** 2 / n2) # Erro-padrão da diferença.
t_calculado = diferenca / erro_padrao # Estatística t.

print("Diferença entre as médias:", round(diferenca, 4))
print("Erro-padrão:", round(erro_padrao, 4))
print("t calculado:", round(t_calculado, 4))

Diferença entre as médias: 3.75
Erro-padrão: 1.8114
t calculado: 2.0702


**Passo 4.** Conferir com a função `ttest_ind`, que já faz tudo de uma vez. A primeira lista informada é `maquina_x` (grupo 1) e a segunda é `maquina_y` (grupo 2).

In [31]:
resultado = stats.ttest_ind(maquina_x, maquina_y, equal_var=False, alternative='two-sided')

print("t (SciPy):", round(resultado.statistic, 4))
print("valor-p:", round(resultado.pvalue, 4))
print("Graus de liberdade:", round(resultado.df, 2))

t (SciPy): 2.0702
valor-p: 0.0575
Graus de liberdade: 13.92


**Passo 5.** Comparar com o valor crítico e tomar a decisão.

In [32]:
alfa = 0.05
t_critico = stats.t.ppf(1 - alfa / 2, resultado.df) # Teste bilateral: alfa dividido entre as duas caudas.
print("t crítico:", round(t_critico, 4))

if resultado.pvalue < alfa:
    print("Decisão: rejeitamos H0.")
else:
    print("Decisão: não rejeitamos H0.")

t crítico: 2.146
Decisão: não rejeitamos H0.


**Interpretação gerencial.** A máquina X produziu, em média, 120,88 peças por hora, e a máquina Y, 117,13, uma diferença de 3,75 peças (t = 2,07; valor-p ≈ 0,058). Ao nível de 5%, **não rejeitamos** $H_0$, por pouco. Se a empresa tivesse adotado $\alpha = 10\%$, a conclusão seria outra, o que mostra que o nível de significância deve ser definido **antes** da análise. Com um resultado limítrofe, a decisão de compra deve pesar também preço, manutenção e consumo de energia, e, se possível, incluir mais turnos de teste.

---
## Síntese da aula

| Exemplo | Teste | $\bar{x}_1 - \bar{x}_2$ | t | gl | valor-p | Decisão ($\alpha$ = 5%) |
|---|---|---|---|---|---|---|
| 1. Aplicativo e loja física | unilateral (maior) | 23,91 | 4,69 | 16,92 | 0,0001 | rejeita $H_0$ |
| 2. Prazo de fornecedores | bilateral | 3,18 | 4,37 | 13,08 | 0,0007 | rejeita $H_0$ |
| 3. Espera nas agências | unilateral (menor) | −0,86 | −3,07 | 12,43 | 0,0047 | rejeita $H_0$ |
| 4. Satisfação remoto e presencial | bilateral | 0,08 | 0,26 | 18,02 | 0,7968 | não rejeita $H_0$ |
| 5. Nova embalagem | unilateral (maior) | 2,00 | 1,38 | 13,68 | 0,0950 | não rejeita $H_0$ |
| 6. Máquinas de estampagem | bilateral | 3,75 | 2,07 | 13,92 | 0,0575 | não rejeita $H_0$ |

**Pareado ou independente?** Pergunte: cada valor de um grupo tem um par específico no outro grupo (mesma pessoa, mesma loja, mesmo produto)? Se sim, use `ttest_rel`. Se os grupos são formados por elementos diferentes, use `ttest_ind`.

**Roteiro para qualquer teste t para amostras independentes em Python:**

1. digite os dados em duas listas, uma para cada grupo;
2. defina qual é o grupo 1 e as hipóteses antes de olhar o resultado;
3. use `stats.ttest_ind(grupo1, grupo2, equal_var=False, alternative=...)`;
4. compare o valor-p com $\alpha$;
5. traduza a decisão estatística em uma recomendação de negócios.

## Exercícios propostos

**Exercício 1.** Um restaurante trabalha com dois aplicativos de entrega. O gerente registrou o tempo de entrega, em minutos, de 9 pedidos feitos pelo aplicativo Rápido e de 8 pedidos feitos pelo aplicativo Expresso.

* Rápido: 32, 28, 35, 30, 27, 33, 31, 29, 34
* Expresso: 38, 35, 41, 33, 37, 40, 36, 39

Ao nível de 5%, o tempo médio de entrega do aplicativo Rápido é menor? Escreva as hipóteses, execute o teste e redija uma recomendação para o gerente.

**Exercício 2.** Uma rede hoteleira comparou as notas de avaliação (0 a 10) dadas por hóspedes de duas de suas unidades.

* Unidade Praia: 8,2; 7,9; 8,5; 8,8; 7,6; 8,1; 8,4; 8,0; 8,3; 7,8
* Unidade Centro: 8,0; 8,6; 7,7; 8,3; 8,1; 7,9; 8,4; 8,2

Ao nível de 5%, há diferença na avaliação média das duas unidades?

Use as células abaixo para resolver.

In [33]:
# Exercício 1


In [34]:
# Exercício 2


## Referências

ANDERSON, D. R.; SWEENEY, D. J.; WILLIAMS, T. A.; CAMM, J. D.; COCHRAN, J. J. *Estatística aplicada à administração e economia*. São Paulo: Cengage Learning. Capítulo sobre inferência estatística para duas populações.

VIRTANEN, P. et al. SciPy 1.0: fundamental algorithms for scientific computing in Python. *Nature Methods*, v. 17, p. 261–272, 2020.

SCIPY. *scipy.stats.ttest_ind*. Documentação oficial. Disponível em: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html.